In [1]:
# -*- coding: utf-8 -*-
"""
安全帽检测程序 — YOLOv8 可视化检测（Jupyter Notebook专用）
支持：单图 / 文件夹 / 摄像头 / 视频
新增：运行时手动输入检测路径、中文无乱码、底部统计文字
"""
import cv2
import os
import numpy as np
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageDraw, ImageFont

# ======================== 中文绘制工具（解决问号乱码） ========================
def put_chinese_text(img, text, pos, font_size=24, color=(255,255,255)):
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    # Windows自带黑体，无需额外字体文件
    font = ImageFont.truetype("simhei.ttf", font_size)
    draw.text(pos, text, font=font, fill=color)
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

# ======================== 【核心配置区，按需修改】 ========================
MODEL_PATH = r"E:\2026暑假实训\runs\detect\helmet_train1-6\weights\best.pt"
CLASS_NAMES = {0: "hat (戴帽)", 1: "person (未戴)"}
COLORS = {
    0: (0, 255, 0),    # 绿色 = 戴安全帽
    1: (0, 0, 255),    # 红色 = 未戴安全帽
}
# 默认检测源（不使用输入框时生效）
DEFAULT_SOURCE = r"E:\2026暑假实训\helmet_yolo\images\val"
CONF_THRESHOLD = 0.3  # 置信度阈值
OUTPUT_DIR = r"E:\2026暑假实训\detect_results"
SAVE_RESULT = True    # 是否保存检测结果
SHOW_IMAGE = True     # 是否在Notebook显示图片（摄像头/视频无效）
USE_INPUT_BOX = True  # True=运行时弹出输入框自定义路径；False=使用上面DEFAULT_SOURCE
# ======================================================================


def draw_boxes(img, results, conf_threshold=0.3):
    """在图片上画检测框和标签（中文无乱码）"""
    if results[0].boxes is None:
        return img

    for box in results[0].boxes:
        conf = float(box.conf[0])
        if conf < conf_threshold:
            continue

        cls_id = int(box.cls[0])
        label = CLASS_NAMES.get(cls_id, f"class_{cls_id}")
        color = COLORS.get(cls_id, (255, 255, 255))

        # 坐标
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        # 画框
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        # 标签文字
        text = f"{label} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        # 标签背景条
        cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        # 中文绘制文字，不再用cv2.putText避免问号
        img = put_chinese_text(img, text, (x1 + 2, y1 - th - 4), font_size=16)

    return img


def detect_image(model, source, conf, save=True, show=False):
    """检测单张图片"""
    results = model(source, conf=conf, verbose=False)
    img = draw_boxes(results[0].orig_img.copy(), results, conf)

    # ========== 底部中文统计文字 ==========
    hats = 0
    persons = 0
    if results[0].boxes is not None:
        hats = sum(1 for b in results[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) >= conf)
        persons = sum(1 for b in results[0].boxes if int(b.cls[0]) == 1 and float(b.conf[0]) >= conf)

    h, w = img.shape[:2]
    if hats + persons > 0:
        text = f"检测到：{hats}人佩戴安全帽，{persons}人未佩戴安全帽"
    else:
        text = "未检测到安全帽相关目标"
    # 黑色底部条保留
    cv2.rectangle(img, (0, h - 35), (w, h), (0, 0, 0), -1)
    # 中文绘制，彻底解决问号
    img = put_chinese_text(img, text, (20, h - 32), font_size=22)
    # ======================================

    # Notebook用matplotlib展示
    if show:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(img_rgb)
        plt.axis("off")
        plt.show()

    if save:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        name = Path(source).name
        out_path = os.path.join(OUTPUT_DIR, f"detected_{name}")
        cv2.imwrite(out_path, img)
        print(f"  已保存: {out_path}")

    # 控制台打印统计
    print(f"  检测到: 戴帽={hats}, 未戴={persons}")
    return img


def detect_folder(model, folder, conf):
    """检测文件夹内所有图片"""
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    files = [f for f in os.listdir(folder)
             if os.path.splitext(f)[1].lower() in exts]

    print(f"检测 {len(files)} 张图片...\n")

    total_hats = total_persons = 0
    for i, fname in enumerate(files, 1):
        fpath = os.path.join(folder, fname)
        print(f"[{i}/{len(files)}] {fname}")
        results = model(fpath, conf=conf, verbose=False)
        if results[0].boxes is not None:
            hats = sum(1 for b in results[0].boxes if int(b.cls[0]) == 0)
            persons = sum(1 for b in results[0].boxes if int(b.cls[0]) == 1)
            total_hats += hats
            total_persons += persons
        detect_image(model, fpath, conf, save=SAVE_RESULT, show=False)

    print(f"\n{'=' * 40}")
    print(f"统计: 总戴帽={total_hats}, 总未戴={total_persons}")


def detect_camera(model, conf):
    """摄像头实时检测（Jupyter单元格内刷新，无独立cv2弹窗）"""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("无法打开摄像头！请检查摄像头是否被其他软件占用")
        return

    print("摄像头已启动，画面在下方刷新，点击工具栏【停止方块】结束检测")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("读取摄像头画面失败，退出")
            break

        results = model(frame, conf=conf, verbose=False)
        frame = draw_boxes(frame, results, conf)

        # ========== 底部实时中文统计文字 ==========
        hats = 0
        persons = 0
        if results[0].boxes is not None:
            hats = sum(1 for b in results[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) >= conf)
            persons = sum(1 for b in results[0].boxes if int(b.cls[0]) == 1 and float(b.conf[0]) >= conf)

        h, w = frame.shape[:2]
        if hats + persons > 0:
            text = f"佩戴:{hats}人 未佩戴:{persons}人"
        else:
            text = "未检测到安全帽"
        cv2.rectangle(frame, (0, h - 35), (w, h), (0, 0, 0), -1)
        frame = put_chinese_text(frame, text, (20, h - 32), font_size=22)
        # ======================================

        # 单元格内刷新画面
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(frame_rgb)
        plt.axis("off")
        plt.show()
        clear_output(wait=True)

    cap.release()
    print("摄像头已关闭")


# ======================== 输入框交互函数 ========================
def get_input_source():
    """弹出输入框，手动输入检测路径"""
    path_input = widgets.Text(
        value=r"E:\2026暑假实训\helmet_yolo\images\val",
        placeholder="输入图片/视频/文件夹路径，摄像头输入0",
        description="检测路径:",
        layout=widgets.Layout(width="800px")
    )
    confirm_btn = widgets.Button(description="开始检测", button_style="success")
    output = widgets.Output()

    def on_click(b):
        with output:
            output.clear_output()
            global run_source
            run_source = path_input.value.strip()
            print(f"已读取输入路径：{run_source}")

    confirm_btn.on_click(on_click)
    display(path_input, confirm_btn, output)

    global run_source
    run_source = None
    while run_source is None:
        pass
    return run_source


# ======================== 主执行逻辑 ========================
if __name__ == "__main__":
    print(f"加载模型权重: {MODEL_PATH}")
    model = YOLO(MODEL_PATH)
    print("模型加载完成\n")

    if USE_INPUT_BOX:
        print("===== 请在下方输入框填写检测路径 =====")
        source = get_input_source()
    else:
        source = DEFAULT_SOURCE

    print(f"当前使用检测源：{source}\n")

    if source == "0":
        detect_camera(model, CONF_THRESHOLD)
    elif os.path.isfile(source):
        ext = os.path.splitext(source)[1].lower()
        if ext in {".mp4", ".avi", ".mov", ".mkv", ".webm"}:
            detect_video(model, source, CONF_THRESHOLD)
        else:
            print(f"开始检测图片: {source}")
            detect_image(model, source, CONF_THRESHOLD, save=SAVE_RESULT, show=SHOW_IMAGE)
    elif os.path.isdir(source):
        detect_folder(model, source, CONF_THRESHOLD)
    else:
        print(f"错误：无效的检测路径 -> {source}")


def detect_video(model, source, conf):
    """检测视频文件"""
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"无法打开视频: {source}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    name = Path(source).stem
    out_path = os.path.join(OUTPUT_DIR, f"{name}_detected.mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    print(f"处理视频 ({total_frames} 帧, {fps} fps)...")
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        results = model(frame, conf=conf, verbose=False)
        frame = draw_boxes(frame, results, conf)

        # 视频底部中文统计
        hats = 0
        persons = 0
        if results[0].boxes is not None:
            hats = sum(1 for b in results[0].boxes if int(b.cls[0]) == 0 and float(b.conf[0]) >= conf)
            persons = sum(1 for b in results[0].boxes if int(b.cls[0]) == 1 and float(b.conf[0]) >= conf)
        h, w = frame.shape[:2]
        if hats + persons > 0:
            text = f"佩戴:{hats}人 未佩戴:{persons}人"
        else:
            text = "未检测到安全帽"
        cv2.rectangle(frame, (0, h - 35), (w, h), (0, 0, 0), -1)
        frame = put_chinese_text(frame, text, (20, h - 32), font_size=22)

        out.write(frame)
        if frame_idx % 30 == 0:
            print(f"  进度: {frame_idx}/{total_frames} ({100 * frame_idx // total_frames}%)")

    cap.release()
    out.release()
    print(f"视频处理完成，已保存: {out_path}")

KeyboardInterrupt: 